In [1]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.http import HttpTool
from autogen_core.models import UserMessage
from autogen_ext.models.ollama import OllamaChatCompletionClient

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
import json


In [3]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [4]:
model=ChatGroq(model="qwen-qwq-32B")
def llm(input):
    model=ChatGroq(model="qwen-qwq-32B")
    output=model.invoke(input)
    return output.content

In [5]:
print(model)

client=<groq.resources.chat.completions.Completions object at 0x112f3afc0> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11218b950> model_name='qwen-qwq-32B' model_kwargs={} groq_api_key=SecretStr('**********')


In [6]:
# model1=OpenAIChatCompletionClient(model="gpt-4o")
# model1

In [7]:
fact_schema = {
    "type": "object",
    "properties": {
        "fact": {
            "type": "string",
            "description": "A factual statement about cats"
        },
        "length": {
            "type": "integer",
            "description": "The length of the fact string"
        }
    },
    "required": ["fact", "length"]
}

http_tool= HttpTool(
    name="base64_decode",
    description="base64 decode a value",
    scheme="https",
    host="catfact.ninja",
    port=443,
    path="/fact",
    method="GET",
    return_type="json",
    json_schema=fact_schema
)


In [8]:
open_router_api_key = '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'

open_router_model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/llama-3.3-nemotron-super-49b-v1:free",
    api_key = open_router_api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)


response = await open_router_model_client.create([UserMessage(content="What is the capital of France?", source="user")])

print(response)

/Users/arunkumar/anaconda3/envs/py312/lib/python3.12/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


finish_reason='stop' content='The answer is quite famous!\n\nThe capital of France is **Paris**. \n\nWould you like to know more about Paris or is there something else I can help you with?' usage=RequestUsage(prompt_tokens=22, completion_tokens=36) cached=False logprobs=None thought=None


In [9]:
open_router_model_client

In [10]:
agent = AssistantAgent(
    name="CatFactsAgent",
    model_client=open_router_model_client,
    system_message='You are a helpful assistant that can provide cat facts using the cat_facts_api tool. Give the result with summary',
    tools=[http_tool],
    reflect_on_tool_use=True
)

In [11]:
# async def main(): 
#     result = await agent.run(task='Give me a random cat fact')
#     print(result.messages)

# await main()


In [12]:
os.environ["SERP_API_KEY"]=os.getenv("SERP_API_KEY")

In [13]:
from langchain_community.utilities import GoogleSerperAPIWrapper

In [14]:
os.environ['SERPER_API_KEY']='03efe53bf8044f01c84e9c0d43a252fc74dc64a7'


search_tool_wrapper =GoogleSerperAPIWrapper(type='search')
#search_tool_wrapper=GoogleSerperAPIWrapper(type='news')

In [17]:
def search_web(query:str) ->str:
    """Search the web for the given query and return the results."""
    try:
        results = search_tool_wrapper.run(query)
        return results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "No results found."  


search_agent = AssistantAgent(
    name="SearchAgent",
    model_client=open_router_model_client,
    tools=[search_web],
    description="An agent that can search the web for information.",
    system_message="You are a helpful assistant that can search the web for information using the search_web tool." \
    "Please make sure that you use the search_web tool to find information before you return the answer.",
    reflect_on_tool_use=True,
)

async def run_serper_search():
    """Run the search agent with a sample query."""
    query = "Who won the IPL in 2025 ?" 
    print(f"Querying: {query}")
    
    
    result = await search_agent.run(task=query)
    print(result.messages[-1].content)


if __name__ == "__main__":
    asyncio.run(run_serper_search())

RuntimeError: asyncio.run() cannot be called from a running event loop